In [5]:
from crewai import Agent, Crew, Process, Task
from crewai.project import CrewBase, agent, crew, task
from dotenv import load_dotenv
import os
from crewai import Agent, Task, Crew, Process, LLM
import os
from langchain_openai import ChatOpenAI
from crewai_tools import MCPServerAdapter
from dotenv import load_dotenv
# from langchain_groq import ChatGroq

import os
from crewai import LLM
from langchain_openai import ChatOpenAI

In [6]:
load_dotenv()

True

In [7]:
from crew_compliance_checker import ComplianceCheckerCrew

In [8]:
model_ids = [
    "groq/compound",
    "groq/compound-mini",
    "llama-3.1-8b-instant",
    "llama-3.3-70b-versatile",
    "meta-llama/llama-4-scout-17b-16e-instruct",
    "meta-llama/llama-prompt-guard-2-22m",
    "meta-llama/llama-prompt-guard-2-86m",
    "openai/gpt-oss-120b",
    "openai/gpt-oss-20b",
    "openai/gpt-oss-safeguard-20b",
    "qwen/qwen3-32b",
]

In [9]:
model_id = model_ids[3]
model_id

'llama-3.3-70b-versatile'

In [10]:
llm = ChatOpenAI(
    openai_api_base="https://api.groq.com/openai/v1",
    openai_api_key=os.environ.get("GROQ_API_KEY"),
    temperature=0,
    model_name=f"groq/{model_id}",
    top_p=1,
    max_retries=3,
    request_timeout=60,
)

In [11]:
crew = ComplianceCheckerCrew(llm=llm, max_crew_rpm=1)

In [12]:
crew.default_llm

ChatOpenAI(output_version=None, client=<openai.resources.chat.completions.completions.Completions object at 0x7f31ca157380>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7f31c9bda7b0>, root_client=<openai.OpenAI object at 0x7f3202d825a0>, root_async_client=<openai.AsyncOpenAI object at 0x7f31c9a6cef0>, model_name='groq/llama-3.3-70b-versatile', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://api.groq.com/openai/v1', openai_proxy=None, request_timeout=60.0, max_retries=3, top_p=1.0, stream_chunk_timeout=120.0)

In [13]:
len(crew.mcp_adapter.tools)

8

In [14]:
reviewed_sql = 'SELECT score FROM table_name_89 WHERE visitor = "toronto" AND record = "29-17-8"'

In [15]:
compliance_task = Task(
    description=f"""
    Check this SQL query for compliance and security issues: {reviewed_sql}
    Report any potential PII exposure or dangerous operations.
    """,
    agent=crew.compliance_checker_agent(),
    expected_output="A compliance report with clear verdict",
)

In [16]:
compliance_crew = Crew(
    agents=[crew.compliance_checker_agent()],
    tasks=[compliance_task],
)

In [17]:
compliance_result = compliance_crew.kickoff()

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Data Security and Compliance Auditor                                                                    │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Check this SQL query for compliance and security issues: SELECT score FROM table_name_89 WHERE visitor =   │
│  "toronto" AND record = "29-17-8"                                                                               │
│      Report any potential PII exposure or dangerous operations.                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Data Security and Compliance Auditor                                                                    │
│                                                                                                                 │
│  Thought: Thought: The given SQL query is a SELECT query, which is generally safe unless it exposes sensitive   │
│  data without filtering. To verify the query's compliance and security, I need to check if the table exists     │
│  and review its schema to ensure it doesn't contain any personally identifiable information (PII) and that the  │
│  query is properly filtered.                                                                                    │
│                                                                                                                 │
│  Using Tool: check_table_exists                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  {                                                                                                              │
│    "table_name": "table_name_89"                                                                                │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  ✓ Table 'table_name_89' exists                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Data Security and Compliance Auditor                                                                    │
│                                                                                                                 │
│  Thought: in the database.                                                                                      │
│  Thought: Since the table exists, I should review its schema to understand the structure and content of the     │
│  table, especially the 'score', 'visitor', and 'record' columns.                                                │
│                                                                                                                 │
│  Using Tool: get_database_schema                                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  {}                                                                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  DATABASE SCHEMA:                                                                                               │
│  ================================================================================                               │
│                                                                                                                 │
│                                                                                                                 │
│  Table: table_name_89                                                                                           │
│  --------------------------------------------------------------------------------                               │
│  Columns:                                                                                                       │
│    - score: TEXT                                                                                                │
│    - visitor: TEXT                                                                                              │
│    - record: TEXT                                                                                               │
│                                                                                                                 │
│  Row count: 16                                                                                                  │
│                                                                                                                 │
│  Sample data (first 3 rows):                                                                                    │
│  score  visitor   record                                                                                        │
│    4-3  toronto  35-15-6                                                                                        │
│    4-2  toronto  18-25-7                                                                                        │
│    6-3 montreal 23-19-10                                                                                        │
│                                                                                                                 │
│  ================================================================================                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Data Security and Compliance Auditor                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  COMPLIANCE REPORT:                                                                                             │
│                                                                                                                 │
│  Query: SELECT score FROM table_name_89 WHERE visitor = "toronto" AND record = "29-17-8"                        │
│                                                                                                                 │
│  VERDICT: PASS                                                                                                  │
│                                                                                                                 │
│  REASON:                                                                                                        │
│  - The query is a SELECT query, which is generally safe.                                                        │
│  - The query filters data based on 'visitor' and 'record', reducing the risk of exposing sensitive data.        │
│  - The 'visitor' column, although it contains team names, does not seem to pose a PII exposure risk in this     │
│  context, as team names are typically public information.                                                       │
│  - The 'score' column, which is the only column being selected, does not contain PII.                           │
│                                                                                                                 │
│  RECOMMENDATIONS:                                                                                               │
│  - Continue to use proper filtering in queries to minimize data exposure risks.                                 │
│  - Regularly review database schema and query logs to ensure ongoing compliance and security.                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [18]:
compliance_report = str(compliance_result).strip()

In [19]:
compliance_report

'COMPLIANCE REPORT:\n\nQuery: SELECT score FROM table_name_89 WHERE visitor = "toronto" AND record = "29-17-8"\n\nVERDICT: PASS\n\nREASON:\n- The query is a SELECT query, which is generally safe.\n- The query filters data based on \'visitor\' and \'record\', reducing the risk of exposing sensitive data.\n- The \'visitor\' column, although it contains team names, does not seem to pose a PII exposure risk in this context, as team names are typically public information.\n- The \'score\' column, which is the only column being selected, does not contain PII.\n\nRECOMMENDATIONS:\n- Continue to use proper filtering in queries to minimize data exposure risks.\n- Regularly review database schema and query logs to ensure ongoing compliance and security.'

In [20]:
compliance_passed = "verdict: pass" in compliance_report.lower()

In [21]:
compliance_passed

True

In [22]:
compliance_result.token_usage.__dict__

{'total_tokens': 3103,
 'prompt_tokens': 2205,
 'cached_prompt_tokens': 0,
 'completion_tokens': 898,
 'successful_requests': 3}